In [7]:
import random

def choose_color(r, b):
    chosen_int = random.randint(1, r+b)
    if chosen_int <= r: return 1
    return 2


# data structure rule: (1,2) red, blue, dictionary
rule = {1: 1, 2: 2}

def compute_prior(r1, b1, r2, b2):
    
    return {
        1: r1 / (r1+b1),
        2: b1 / (r1+b1)
    }

def compute_likelihood(r1, b1, r2, b2):
    new_urn_2 = r2 + b2 + 1
    return {
        1: {1: (r2 + 1) / new_urn_2, 2: b2 / new_urn_2}, # meaning urn 1 ball was red
        2: {1: r2 / new_urn_2,       2: (b2 + 1) / new_urn_2}       
    }

def compute_posteriors(r1, b1, r2, b2):
    prior = compute_prior(r1, b1, r2, b2)
    likelihood = compute_likelihood(r1, b1, r2, b2)
    p_joint = {
        h: {y: prior[h] * likelihood[h][y] for y in [1, 2]}  for h in [1, 2]
    }
    
    p_y = {
        y: sum(p_joint[h][y] for h in [1, 2])
        for y in [1, 2]
    }
    
    return {
        y: {h: p_joint[h][y] / p_y[y] for h in [1, 2]}
        for y in [1, 2]
    }

def decision_rules(r1, b1, r2, b2):
    prior = compute_prior(r1, b1, r2, b2)
    posteriors = compute_posteriors(r1, b1, r2, b2)
    likelihood = compute_likelihood(r1, b1, r2, b2)
    p_joint = {
        h: {y: prior[h] * likelihood[h][y] for y in [1, 2]}
        for h in [1, 2]
    }


    rule_ml = {
        y: (1 if likelihood[1][y] >= likelihood[2][y] else 2) 
        for y in (1, 2)
    }

    rule_map = {
        y: (1 if p_joint[1][y] >= p_joint[2][y] else 2) 
        for y in (1, 2)
    }

    return rule_ml, rule_map

def compute_errors(r1, b1, r2, b2):
    prior = compute_prior(r1, b1, r2, b2)
    posteriors = compute_posteriors(r1, b2, r2, b2)
    likelihood = compute_likelihood(r1, b1, r2, b2)
    p_joint = {
        h: {y: prior[h] * likelihood[h][y] for y in [1, 2]}
        for h in [1, 2]

    }
    rule_ml, rule_map = decision_rules(r1, b1, r2, b2)
    p_err_ml = 1.0 - sum(p_joint[rule_ml[y]][y] for y in [1, 2])
    p_err_map = 1.0 - sum(p_joint[rule_map[y]][y] for y in [1, 2])
    return p_err_ml, p_err_map




In [ ]:
# Simulation Cell
# Time Efficient

import numpy as np
"""
This following approach is efficient because it uses no loops to generate random numbers over 10k, instead using numpy randN
also the tables use numpy arrays which are more time efficient than python dictionaries since we are doing multi-item lookups;
faster than looping through a dict
"""


def simulate_experiment(r1, b1, r2, b2, rule_ml, rule_map, N=100000):
    p_red1 = r1 / (r1 + b1)
    urn1 = np.where(np.random.rand(N) < p_red1, 1, 2)

    total2 = r2 + b2 + 1
    p_red2 = np.where(urn1 == 1, (r2 + 1) / total2, r2 / total2)
    urn2 = np.where(np.random.rand(N) < p_red2, 1, 2)

    # Index 1 gets rule[1], index 2 gets rule[2]
    ml_table = np.array([0, rule_ml[1], rule_ml[2]])
    map_table = np.array([0, rule_map[1], rule_map[2]])

    guess_ml = ml_table[urn2]
    guess_map = map_table[urn2]

    # Step 4: Fraction of errors
    err_ml = np.mean(guess_ml != urn1)
    err_map = np.mean(guess_map != urn1)

    return err_ml, err_map


# cases (part d)
cases = [
    ("case1",   7, 3, 8, 2),
    ("case2",  4, 6, 7, 3),
    ("case3", 7, 3, 3, 7),
    ("case4",  4, 6, 4, 6),
]


for name, r1, b1, r2, b2 in cases:
    rule_ml, rule_map = decision_rules(r1, b1, r2, b2)
    theoretical_ml, theoretical_map = compute_errors(r1, b1, r2, b2)
    sim_ml, sim_map = simulate_experiment(r1, b1, r2, b2, rule_ml, rule_map)

    print(f"{name}\n")
    print(f"ML  Rule: {rule_ml}; Theo Err: {theoretical_ml}; Sim Err: {sim_ml}")
    print(f"MAP Rule: {rule_map}; Theo Err: {theoretical_map}; Sim Err: {sim_map}\n")

"""Results"""

"""
The experimental results mostly confirm exactly the theoretical errors in both cases; they only differ slightly after 3 decimal places
and majorly only after 6-7 decimal places which we can attribute as computer computational error (floating point precision)

MAP produces lower error in every single instance, since ML always choioses the observed color, since adding that color increases likelihood, but MAP takes into account the conditions of Urn 1, which is more important.

MAP is significantly better when there are large differences in B/R between urns such as case 3 which produces the biggest error gap.
"""

case1

ML  Rule: {1: 1, 2: 2}; Theo Err: 0.34545454545454546; Sim Err: 0.3464
MAP Rule: {1: 1, 2: 1}; Theo Err: 0.30000000000000004; Sim Err: 0.30121

case2

ML  Rule: {1: 1, 2: 2}; Theo Err: 0.49090909090909085; Sim Err: 0.48995
MAP Rule: {1: 2, 2: 2}; Theo Err: 0.4; Sim Err: 0.39693

case3

ML  Rule: {1: 1, 2: 2}; Theo Err: 0.5272727272727273; Sim Err: 0.52703
MAP Rule: {1: 1, 2: 1}; Theo Err: 0.30000000000000004; Sim Err: 0.30155

case4

ML  Rule: {1: 1, 2: 2}; Theo Err: 0.4363636363636364; Sim Err: 0.43678
MAP Rule: {1: 2, 2: 2}; Theo Err: 0.4; Sim Err: 0.40133

